# `panorama.units` — Demo

Worked examples of the **`units`** module of PANORAMA: Doppler shifts, vacuum/air wavelengths, resolution limits, characteristic stellar velocities and derived stellar parameters.

* **Examples 1-4** — wavelength conversions: non-relativistic and relativistic Doppler shifts (including the ≥ 1 % c warning and the error for unphysical shifts), vacuum/air wavelengths (including the out-of-range warning) and resolution limits.
* **Examples 5-6** — characteristic velocities: thermal, Doppler and escape velocities.
* **Examples 7-11** — stellar parameters: Stefan-Boltzmann law, bolometric magnitudes, a solar sanity check, the centrifugal, radiative and combined corrections of an effective log g and their effect on the spectroscopic mass, and the spectroscopic luminosity.
* **Examples 12-13** — the Eddington factor of a non-rotating star and the full workflow for a rotating one (fitted log g → `logg_centrifugal` → `eddington`), including a star beyond the ΩΓ limit.

**Gravity notation:** `g_eff` is the effective gravity, `g_grav` the gravitational (Newtonian) one, and `g_cen` and `g_rad` the centrifugal and radiative accelerations (g_grav = g_eff + g_cen + g_rad).

The notebook needs no input data and no internet access, and writes no files. Warnings are captured and printed, so that the output is self-explanatory. Run the cells in order (*Run All*).

## Imports

Python packages and PANORAMA functions.

In [1]:
import warnings

from panorama import (
                      air2vac,  # WAVELENGTH
                      eddington,  # STELLAR PARAMETERS
                      kmps2R,  # VELOCITY
                      kmps2wl,  # WAVELENGTH
                      kmps2wl_rel,  # WAVELENGTH
                      logg_centrifugal,  # STELLAR PARAMETERS
                      logg_rad_centrifugal,  # STELLAR PARAMETERS
                      logg_radiative,  # STELLAR PARAMETERS
                      loggradius2mass,  # STELLAR PARAMETERS
                      logL2Mbol,  # STELLAR PARAMETERS
                      logLteff2radius,  # STELLAR PARAMETERS
                      Mbol2logL,  # STELLAR PARAMETERS
                      R2kmps,  # VELOCITY
                      spec_luminosity,  # STELLAR PARAMETERS
                      teffradius2logL,  # STELLAR PARAMETERS
                      vac2air,  # WAVELENGTH
                      vescape,  # VELOCITY
                      vthermal,  # VELOCITY
                      wl2kmps,  # WAVELENGTH
                      wl2kmps_rel,  # WAVELENGTH
                     )

	PANORAMA v1.2.1 loaded...
		- To see the available functions, run:
			panorama.functions()          # Lists all functions, grouped by submodule
			panorama.functions('obs')     # Lists the functions of a single submodule
		- To see the docstring of a specific function (e.g. 'visibility'), run:
			help(panorama.visibility)
		- To silence this message, set the environment variable PANORAMA_QUIET=1



## Example 1: Non-relativistic Doppler conversion (Si III 4553)

Non-relativistic Doppler conversion and velocity widths in wavelength units.

In [2]:
wl0_siIII     = 4552.62  # Si III 4553 rest wavelength (air) [Å]
dwl           = 0.5  # Observed shift [Å]
rv            = wl2kmps(dwl, wl0_siIII)
recovered_dwl = kmps2wl(rv, wl0_siIII)


print(f"\nΔλ = {dwl:.3f} Å at λ0 = {wl0_siIII:.2f} Å")
print(f"\tRadial velocity     = {rv:.3f} km/s")
print(f"\tRecovered Δλ        = {recovered_dwl:.3f} Å")

print("\nVelocity widths expressed in Å at the same line:")
for width in [10., 50., 150., 300.]:  # E.g. v sin i or macroturbulence [km/s]
    print(f"\t{width:6.0f} km/s  →  {kmps2wl(width, wl0_siIII):6.3f} Å")


Δλ = 0.500 Å at λ0 = 4552.62 Å
	Radial velocity     = 32.925 km/s
	Recovered Δλ        = 0.500 Å

Velocity widths expressed in Å at the same line:
	    10 km/s  →   0.152 Å
	    50 km/s  →   0.759 Å
	   150 km/s  →   2.278 Å
	   300 km/s  →   4.556 Å


## Example 2: Relativistic vs. non-relativistic Doppler conversion

Relativistic vs. non-relativistic Doppler conversion, warning and unphysical shifts.

In [3]:
wl0     = 5000.  # Rest wavelength [Å]
dwl_big = 100.  # Large shift: |v| ≈ 2% of c [Å]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    rv_nonrel = wl2kmps(dwl_big, wl0)  # Triggers the >= 1% c warning on purpose
rv_rel            = wl2kmps_rel(dwl_big, wl0)
recovered_dwl_rel = kmps2wl_rel(rv_rel, wl0)


print(f"\nΔλ = {dwl_big:.1f} Å at λ0 = {wl0:.1f} Å")
print(f"\tNon-relativistic velocity = {rv_nonrel:9.2f} km/s")
print(f"\tRelativistic velocity     = {rv_rel:9.2f} km/s")
print(f"\tDifference                = {rv_nonrel - rv_rel:9.2f} km/s")
print(f"\tRecovered Δλ (relativistic round trip) = {recovered_dwl_rel:.6f} Å")
for w in caught:
    print(f"\tWarning: {w.message}")

print("\nUnphysical inputs are rejected:")
try:
    wl2kmps_rel(-6000., wl0)  # Observed wavelength would be negative
except ValueError as e:
    print(f"\tValueError: {e}")
try:
    kmps2wl_rel(4.e5, wl0)  # Faster than light
except ValueError as e:
    print(f"\tValueError: {e}")


Δλ = 100.0 Å at λ0 = 5000.0 Å
	Non-relativistic velocity =   5995.85 km/s
	Relativistic velocity     =   5935.90 km/s
	Difference                =     59.95 km/s
	Recovered Δλ (relativistic round trip) = 100.000000 Å

Unphysical inputs are rejected:
	ValueError: Wavelength shift 'dwl' must be greater than -wl0 (the observed wavelength wl0 + dwl must be positive). Got dwl = -6000.0 Å and wl0 = 5000.0 Å.
	ValueError: Radial velocity 'dkmps' = 400000.0 km/s implies |β| = 1.3342563807926082 >= 1 (faster than or equal to the speed of light); the relativistic Doppler formula is undefined.


## Example 3: Vacuum ↔ air wavelength conversion (Edlén 1966)

Vacuum ↔ air wavelengths (Edlén 1966), round trip and out-of-range warning.

In [4]:
lines_vac = {  # Vacuum wavelengths [Å]
             'H-beta'  : 4862.68,
             'He I'    : 5877.25,
             'H-alpha' : 6564.61,
            }


print(f"\n\t{'Line':<8} {'λ_vac [Å]':>10} {'λ_air [Å]':>10} {'Δλ [Å]':>7} {'Δv [km/s]':>10} {'Round trip [Å]':>15}")
for name, wl_vac in lines_vac.items():
    wl_air = vac2air(wl_vac)
    print(
          f"\t{name:<8} {wl_vac:10.2f} {wl_air:10.2f} {wl_vac - wl_air:7.3f} "
          f"{wl2kmps(wl_vac - wl_air, wl_air):10.1f} {air2vac(wl_air) - wl_vac:15.1e}"
         )
print("\t(air2vac is the exact inverse of vac2air to floating-point precision)")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    vac2air(1500.)  # Far-UV: outside the validity range of the formula
print("\nOutside 2000-10000 Å:")
for w in caught:
    print(f"\tWarning: {w.message}")


	Line      λ_vac [Å]  λ_air [Å]  Δλ [Å]  Δv [km/s]  Round trip [Å]
	H-beta      4862.68    4861.32   1.358       83.7         0.0e+00
	He I        5877.25    5875.62   1.628       83.1         0.0e+00
	H-alpha     6564.61    6562.80   1.813       82.8         0.0e+00
	(air2vac is the exact inverse of vac2air to floating-point precision)

Outside 2000-10000 Å:


## Example 4: Resolving power ↔ velocity resolution limit

In [5]:
print("\nResolving power → smallest resolvable broadening:")
for R in [5000, 20000, 46000, 85000]:
    print(f"\tR = {R:6d}  →  Δv = {R2kmps(R):6.2f} km/s")

print("\nBroadening → required resolving power:")
for v in [5., 10., 20., 50.]:
    print(f"\tΔv = {v:5.1f} km/s  →  R = {kmps2R(v):8.0f}")


Resolving power → smallest resolvable broadening:
	R =   5000  →  Δv =  79.15 km/s
	R =  20000  →  Δv =  19.79 km/s
	R =  46000  →  Δv =   8.60 km/s
	R =  85000  →  Δv =   4.66 km/s

Broadening → required resolving power:
	Δv =   5.0 km/s  →  R =    79145
	Δv =  10.0 km/s  →  R =    39573
	Δv =  20.0 km/s  →  R =    19786
	Δv =  50.0 km/s  →  R =     7915


## Example 5: Thermal and Doppler velocities

Thermal and total Doppler velocities of several ions.

In [6]:
teff_th  = 30000.  # [K]
vmic     = 10.  # Microturbulence [km/s]
elements = {  # Atomic masses [u]
            'H'  : 1.008,
            'He' : 4.003,
            'N'  : 14.007,
            'Si' : 28.086,
            'Fe' : 55.845,
           }


print(f"\n\t{'Ion':<4} {'v_th [km/s]':>12} {'v_D [km/s]':>11} {'Δλ_D at 4553 Å':>15}")
for name, A in elements.items():
    res = vthermal(teff_th, A, vmic)
    print(f"\t{name:<4} {res['v_th']:12.2f} {res['v_doppler']:11.2f} {kmps2wl(res['v_doppler'], wl0_siIII):13.3f} Å")
print("\t(Macroturbulence is not added: it is applied by convolution, like rotation)")


	Ion   v_th [km/s]  v_D [km/s]  Δλ_D at 4553 Å
	H           22.25       24.39         0.370 Å
	He          11.16       14.99         0.228 Å
	N            5.97       11.65         0.177 Å
	Si           4.21       10.85         0.165 Å
	Fe           2.99       10.44         0.158 Å
	(Macroturbulence is not added: it is applied by convolution, like rotation)


## Example 6: Photospheric escape velocity (M = 20 Msun, R = 8 Rsun)

Newtonian and radiation-corrected escape velocities.

In [7]:
print(f"\n\t{'Γ_e':>5} {'v_esc [km/s]':>13} {'v_esc,eff [km/s]':>17} {'v_∞ ≈ 2.6 v_esc,eff':>20}")
for gamma_e in [0., 0.25, 0.5]:
    esc = vescape(mass=20., radius=8., gamma=gamma_e)
    print(f"\t{esc['gamma']:5.2f} {esc['v_esc']:13.1f} {esc['v_esc_eff']:17.1f} {2.6 * esc['v_esc_eff']:15.0f} km/s")
print("\t(v_∞ / v_esc,eff ≈ 2.6 on the hot side of the bi-stability jump)")


	  Γ_e  v_esc [km/s]  v_esc,eff [km/s]  v_∞ ≈ 2.6 v_esc,eff
	 0.00         976.6             976.6            2539 km/s
	 0.25         976.6             845.8            2199 km/s
	 0.50         976.6             690.6            1796 km/s
	(v_∞ / v_esc,eff ≈ 2.6 on the hot side of the bi-stability jump)


## Example 7: Stefan-Boltzmann law and bolometric magnitude

In [8]:
teff_ex   = 30000.  # [K]
radius_ex = 10.  # [Rsun]
logL      = teffradius2logL(teff_ex, radius_ex)
Mbol      = logL2Mbol(logL)


print(f"\nTeff = {teff_ex:.0f} K, R = {radius_ex:.1f} Rsun")
print(f"\tlog(L/Lsun) = {logL:.3f}")
print(f"\tMbol        = {Mbol:.2f} mag")

print("\nRecovering the inputs:")
print(f"\tlog(L/Lsun) from Mbol = {Mbol2logL(Mbol):.3f}")
print(f"\tR from logL and Teff  = {logLteff2radius(logL, teff_ex):.3f} Rsun")


Teff = 30000 K, R = 10.0 Rsun
	log(L/Lsun) = 4.863
	Mbol        = -7.42 mag

Recovering the inputs:
	log(L/Lsun) from Mbol = 4.863
	R from logL and Teff  = 10.000 Rsun


## Example 8: Solar sanity check

Sanity check with the Sun (IAU 2015 nominal values).

In [9]:
print(f"\n\tlog(L/Lsun)  (Teff = 5772 K, R = 1 Rsun)    = {teffradius2logL(5772., 1.):+.4f}")
print(f"\tM/Msun       (logg = 4.438, R = 1 Rsun)     = {loggradius2mass(4.438, 1.):.4f}")
print(f"\tlog(ℒ/ℒsun)  (Teff = 5772 K, logg = 4.438)  = {spec_luminosity(5772., 4.438):+.4f}")
print(f"\tMbol         (log(L/Lsun) = 0)              = {logL2Mbol(0.):.2f} mag")


	log(L/Lsun)  (Teff = 5772 K, R = 1 Rsun)    = -0.0000
	M/Msun       (logg = 4.438, R = 1 Rsun)     = 0.9998
	log(ℒ/ℒsun)  (Teff = 5772 K, logg = 4.438)  = +0.0001
	Mbol         (log(L/Lsun) = 0)              = 4.74 mag


## Example 9: Centrifugal correction of log g

Centrifugal correction of log g and its effect on the spectroscopic mass.

In [10]:
logg_fit  = 3.5  # Fitted (effective) gravity
radius_cf = 10.  # [Rsun]
mass_fit  = loggradius2mass(logg_fit, radius_cf)


print(f"\n\t{'vsini [km/s]':>13} {'g_cen [cm/s2]':>14} {'logg_cen':>9} {'logg_grav':>10} {'Δlogg [dex]':>12} {'M_spec [Msun]':>14}")
for vsini in [0., 50., 150., 250., 350.]:
    res = logg_centrifugal(logg=logg_fit, vsini=vsini, radius=radius_cf)
    print(
          f"\t{vsini:13.0f} {res['g_cen']:14.1f} {res['logg_cen']:9.3f} {res['logg_grav']:10.3f} "
          f"{res['dlogg']:12.3f} {loggradius2mass(res['logg_grav'], radius_cf):14.2f}"
         )
print("\t(logg_cen = -inf for vsini = 0)")
print(f"\t(Using the fitted logg, the spectroscopic mass would be {mass_fit:.2f} Msun for every v sin i)")


	 vsini [km/s]  g_cen [cm/s2]  logg_cen  logg_grav  Δlogg [dex]  M_spec [Msun]
	            0            0.0      -inf      3.500        0.000          11.53
	           50           35.9     1.556      3.505        0.005          11.66
	          150          323.4     2.510      3.542        0.042          12.71
	          250          898.4     2.953      3.609        0.109          14.81
	          350         1760.8     3.246      3.692        0.192          17.95
	(logg_cen = -inf for vsini = 0)
	(Using the fitted logg, the spectroscopic mass would be 11.53 Msun for every v sin i)


## Example 10: Radiative and combined corrections of log g

Radiative and combined (radiative + centrifugal) corrections of an effective log g.

In [11]:
logg_eff_in = 3.0  # Effective gravity that does NOT include radiation pressure
teff_rc     = 30000.  # [K]
vsini_rc    = 250.  # [km/s]
radius_rc   = 12.  # [Rsun]

rad  = logg_radiative(logg=logg_eff_in, teff=teff_rc)
both = logg_rad_centrifugal(
                            logg=logg_eff_in,
                            teff=teff_rc,
                            vsini=vsini_rc,
                            radius=radius_rc
                           )


print(f"\nlogg_eff = {logg_eff_in:.2f}, Teff = {teff_rc:.0f} K, vsini = {vsini_rc:.0f} km/s, R = {radius_rc:.0f} Rsun")
print(f"\n\t{'Correction':<24} {'g_rad':>8} {'g_cen':>8} {'g_grav':>8} {'logg_grav':>10} {'Δlogg':>7}")
print(f"\t{'Radiative':<24} {rad['g_rad']:8.1f} {'—':>8} {rad['g_grav']:8.1f} {rad['logg_grav']:10.3f} {rad['dlogg']:7.3f}")
print(f"\t{'Radiative + centrifugal':<24} {both['g_rad']:8.1f} {both['g_cen']:8.1f} {both['g_grav']:8.1f} {both['logg_grav']:10.3f} {both['dlogg']:7.3f}")
print("\t(accelerations in cm/s2; corrections in dex are not additive, accelerations are)")
print(f"\tlogg_rad = {both['logg_rad']:.3f}, logg_cen = {both['logg_cen']:.3f}")
print("\nCaveat: FASTWIND, CMFGEN, TLUSTY and ATLAS already include radiation pressure;")
print("\tfor their fitted gravities use 'logg_centrifugal' alone (no radiative correction).")


logg_eff = 3.00, Teff = 30000 K, vsini = 250 km/s, R = 12 Rsun

	Correction                  g_rad    g_cen   g_grav  logg_grav   Δlogg
	Radiative                   520.9        —   1520.9      3.182   0.182
	Radiative + centrifugal     520.9    748.6   2269.5      3.356   0.356
	(accelerations in cm/s2; corrections in dex are not additive, accelerations are)
	logg_rad = 2.717, logg_cen = 2.874

Caveat: FASTWIND, CMFGEN, TLUSTY and ATLAS already include radiation pressure;
	for their fitted gravities use 'logg_centrifugal' alone (no radiative correction).


## Example 11: Spectroscopic luminosity ℒ = Teff^4 / g

Spectroscopic luminosity ℒ = Teff^4 / g (sHRD).

In [12]:
print(f"\n\t{'Teff [K]':>9} {'logg_grav':>10} {'log(ℒ/ℒsun)':>12}")
for teff_s, logg_s in [(40000., 3.8), (30000., 3.2), (20000., 2.3), (15000., 2.0)]:
    print(f"\t{teff_s:9.0f} {logg_s:10.2f} {spec_luminosity(teff_s, logg_s):12.3f}")


	 Teff [K]  logg_grav  log(ℒ/ℒsun)
	    40000       3.80        4.001
	    30000       3.20        4.101
	    20000       2.30        4.297
	    15000       2.00        4.097


## Example 12: Eddington factor of a non-rotating star

Eddington factor and accelerations of a non-rotating supergiant.

In [13]:
star = eddington(
                 teff=30000.,
                 logg=3.0,
                 radius=12.
                )


print(f"\nTeff = {star['teff']:.0f} K, logg_grav = {star['logg']:.2f}, R = {star['radius']:.1f} Rsun, X = {star['H_frac']:.2f}")
print(f"\tSpectroscopic mass = {star['mass']:.2f} Msun")
print(f"\tL_bol              = {star['L_bol']:.3e} Lsun")
print(f"\tL_edd              = {star['L_edd']:.3e} Lsun")
print(f"\tΓ_e                = {star['gamma_e']:.3f}")

print("\nSurface accelerations [cm/s2]:")
print(f"\tg_grav = {star['g_grav']:9.2f}")
print(f"\tg_rad  = {star['g_rad']:9.2f}")
print(f"\tg_cen  = {star['g_cen']:9.2f}  (vsini = 0)")
print(f"\tg_eff  = {star['g_eff']:9.2f}  → logg_eff = {star['logg_eff']:.3f}")
print(f"\tEddington limit reached: {star['eddington_limit']}, ΩΓ limit reached: {star['omega_gamma_limit']}")

print("\nWith an independent (e.g. evolutionary) mass of 30 Msun:")
star_m = eddington(teff=30000., logg=3.0, radius=12., mass=30.)
print(f"\tΓ_e = {star_m['gamma_e']:.3f}  (g_rad / g_grav = {star_m['g_rad'] / star_m['g_grav']:.3f}: no longer equal, see Notes)")


Teff = 30000 K, logg_grav = 3.00, R = 12.0 Rsun, X = 0.70
	Spectroscopic mass = 5.25 Msun
	L_bol              = 1.051e+05 Lsun
	L_edd              = 2.017e+05 Lsun
	Γ_e                = 0.521

Surface accelerations [cm/s2]:
	g_grav =   1000.00
	g_rad  =    520.90
	g_cen  =      0.00  (vsini = 0)
	g_eff  =    479.10  → logg_eff = 2.680
	Eddington limit reached: False, ΩΓ limit reached: False

With an independent (e.g. evolutionary) mass of 30 Msun:
	Γ_e = 0.091  (g_rad / g_grav = 0.521: no longer equal, see Notes)


## Example 13: Eddington factor and ΩΓ limit of a rotating star

Rotating star: fitted log g → logg_centrifugal → eddington, and a star beyond the ΩΓ limit.

In [14]:
teff_rot   = 22000.  # [K]
logg_rot   = 3.3  # Fitted (effective) gravity
radius_rot = 20.  # [Rsun]
vsini_rot  = 250.  # [km/s]

corrected = logg_centrifugal(logg=logg_rot, vsini=vsini_rot, radius=radius_rot)
rotating  = eddington(
                      teff=teff_rot,
                      logg=corrected['logg_grav'],  # Gravitational gravity: avoids subtracting g_cen twice
                      radius=radius_rot,
                      vsini=vsini_rot
                     )


print(f"\nTeff = {teff_rot:.0f} K, fitted logg = {logg_rot:.2f}, R = {radius_rot:.0f} Rsun, vsini = {vsini_rot:.0f} km/s")
print(f"\tlogg_grav (logg_centrifugal)  = {corrected['logg_grav']:.3f}  (+{corrected['dlogg']:.3f} dex)")
print(f"\tSpectroscopic mass            = {rotating['mass']:.2f} Msun")
print(f"\tΓ_e                           = {rotating['gamma_e']:.3f}")
print(f"\tg_grav, g_cen, g_rad [cm/s2]  = {rotating['g_grav']:.1f}, {rotating['g_cen']:.1f}, {rotating['g_rad']:.1f}")
print(f"\tg_eff                         = {rotating['g_eff']:.1f} cm/s2  → logg_eff = {rotating['logg_eff']:.3f}")
print(f"\tCheck: 10**logg_fit - g_rad   = {10**logg_rot - rotating['g_rad']:.1f} cm/s2 (equal to g_eff)")
print(f"\tΩΓ limit reached: {rotating['omega_gamma_limit']}")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    critical = eddington(
                         teff=teff_rot,
                         logg=3.0,
                         radius=radius_rot,
                         vsini=600.
                        )  # Centrifugal + radiative support exceed gravity

print("\nSame star with logg_grav = 3.0 and vsini = 600 km/s:")
print(f"\tg_eff = {critical['g_eff']:.1f} cm/s2, logg_eff = {critical['logg_eff']}")
print(f"\tEddington limit reached: {critical['eddington_limit']}, ΩΓ limit reached: {critical['omega_gamma_limit']}")
for w in caught:
    print(f"\tWarning: {w.message}")


Teff = 22000 K, fitted logg = 3.30, R = 20 Rsun, vsini = 250 km/s
	logg_grav (logg_centrifugal)  = 3.388  (+0.088 dex)
	Spectroscopic mass            = 35.66 Msun
	Γ_e                           = 0.062
	g_grav, g_cen, g_rad [cm/s2]  = 2444.5, 449.2, 150.6
	g_eff                         = 1844.6 cm/s2  → logg_eff = 3.266
	Check: 10**logg_fit - g_rad   = 1844.6 cm/s2 (equal to g_eff)
	ΩΓ limit reached: False

Same star with logg_grav = 3.0 and vsini = 600 km/s:
	g_eff = -1738.0 cm/s2, logg_eff = nan
	Eddington limit reached: False, ΩΓ limit reached: True
